# Verificación de la capa Silver — Iceberg

Revisamos la tabla `nessie.silver.yellow_taxi` tras la limpieza:
referencias Nessie, metadatos Iceberg, snapshots, particionado y calidad de datos.
También comparamos contra bronze para validar las reglas de limpieza.

In [26]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie
from pyspark.sql.functions import (
    col, count, year as spark_year, month as spark_month,
    sum as spark_sum, round as spark_round, when,
)

spark = get_spark("verify-silver-iceberg")
spark.version

'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

In [27]:
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags     = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:

  * Branch bronze                                   805801d8c412
  * Branch main                                     8c8de63a2a64
  * Branch silver                                   deb5563288bb
    Tag    bronze-taxi_zone_lookup-2026             805801d8c412
    Tag    bronze-yellow_taxi-2019                  7181dde00761
    Tag    bronze-yellow_taxi-2020                  ada5cb129398
    Tag    bronze-yellow_taxi-2022                  715e92428de5
    Tag    bronze-yellow_taxi-2023                  e1055846a26e
    Tag    bronze-yellow_taxi-2025                  251fb659d921
    Tag    gold-pickup_demand_zone_hour-2019        5debea82efe3
    Tag    gold-pickup_demand_zone_hour-2020        e26ad87024b7
    Tag    gold-pickup_demand_zone_hour-2022        2de8088e00fe
    Tag    gold-pickup_demand_zone_hour-2023        4701085a3521
    Tag    gold-pickup_demand_zone_hour-2025        8c8de63a2a64
    Tag    gold-yellow_taxi_trips-2019              1c3f561c4115
    T

In [28]:
# Switch to silver branch for the rest of the notebook
nessie.set_spark_branch(spark, "silver")

spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

+---------+
|namespace|
+---------+
|silver   |
+---------+

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|silver   |taxi_zone_lookup|false      |
|silver   |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [29]:
spark.sql("DESCRIBE EXTENDED nessie.silver.yellow_taxi").show(250, truncate=False)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                 |comment|
+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|VendorID                     |bigint                                          

In [30]:
spark.sql("SHOW TBLPROPERTIES nessie.silver.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |3856894123753471135                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |deb5563288bbd6aee779fbc8d2192fc576de698409dc9bb809edc43d2ad461dc|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [31]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                        |summary                                                    

In [32]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-10 11:00:24.007|1934986932255912622|NULL               |true               |
|2026-05-10 11:01:44.621|410107661762819863 |1934986932255912622|true               |
|2026-05-10 11:03:14.01 |6585008059788151588|410107661762819863 |true               |
|2026-05-10 11:04:30.774|6168256879851663963|6585008059788151588|true               |
|2026-05-10 11:05:47.689|8189997092404486225|6168256879851663963|true               |
|2026-05-10 11:06:45.984|4825062742048773961|8189997092404486225|true               |
|2026-05-10 11:07:41.83 |7746239099242298720|4825062742048773961|true               |
|2026-05-10 11:08:41.529|6602558338879100740|7746239099242298720|true               |
|2026-05-10 11:09:47.822|4547372153792017001|660255833

## 3b. Crecimiento de registros por tag

Cada tag `silver-yellow_taxi-{año}` captura el estado de `silver` tras procesar
todos los meses de ese año. Comparamos también con el tag equivalente en bronze.

In [33]:
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

silver_tags = [
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^silver-yellow_taxi-\d{4}$"))
           .collect()
]
silver_tags.sort()

# Bronze ingiere ficheros completos del fuente que arrastran filas con
# pickup en años fuera del scope (2001, 2018, ...). Silver las descarta
# por regla, así que para que retention_pct compare manzanas con manzanas
# filtramos bronze al mismo conjunto de años válidos.
VALID_YEARS = [2019, 2020, 2022, 2023, 2025]

tag_pattern = re.compile(r"silver-yellow_taxi-(\d{4})")
rows = []
prev_silver = 0
for tag in silver_tags:
    year = int(tag_pattern.match(tag).group(1))

    nessie.set_spark_branch(spark, tag)
    silver_total = spark.table("nessie.silver.yellow_taxi").count()

    bronze_tag = f"bronze-yellow_taxi-{year}"
    try:
        nessie.set_spark_branch(spark, bronze_tag)
        bronze_total = (
            spark.table("nessie.bronze.yellow_taxi")
                 .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
                 .count()
        )
    except Exception:
        bronze_total = None

    retention = round(silver_total / bronze_total * 100, 2) if bronze_total else None
    rows.append((tag, year, bronze_total, silver_total, silver_total - prev_silver, retention))
    prev_silver = silver_total

nessie.set_spark_branch(spark, "silver")

growth_df = spark.createDataFrame(
    rows,
    ["tag", "year", "bronze_records", "silver_records", "records_added", "retention_pct"]
)
growth_df.show(truncate=False)

+-----------------------+----+--------------+--------------+-------------+-------------+
|tag                    |year|bronze_records|silver_records|records_added|retention_pct|
+-----------------------+----+--------------+--------------+-------------+-------------+
|silver-yellow_taxi-2019|2019|22612028      |21926817      |21926817     |96.97        |
|silver-yellow_taxi-2020|2020|38324009      |37125328      |15198511     |96.87        |
|silver-yellow_taxi-2022|2022|47395196      |45665180      |8539852      |96.35        |
|silver-yellow_taxi-2023|2023|56779665      |54573946      |8908766      |96.12        |
|silver-yellow_taxi-2025|2025|67977668      |57423106      |2849160      |84.47        |
+-----------------------+----+--------------+--------------+-------------+-------------+



## 4. Particionado y ficheros

In [34]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{600}    |0      |6201962     |1         |94212089                     |0                           |0                         |0                           |0                         |2026-05-10 11:04:30.774|6168256879851663963     |
|{601}    |0      |6103376     |1         |92685401         

In [35]:
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df = spark.sql("SELECT * FROM nessie.silver.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format", "record_count", "file_size_in_bytes"
).show(100, truncate=80)

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/silver/yellow_taxi_d1a45ac5-5268-493c-818b-acba9f2b22cf/data/...|    PARQUET|     2849160|          44214317|
|      0|s3a://warehouse/silver/yellow_taxi_d1a45ac5-5268-493c-818b-acba9f2b22cf/data/...|    PARQUET|     3226937|          51399336|
|      0|s3a://warehouse/silver/yellow_taxi_d1a45ac5-5268-493c-818b-acba9f2b22cf/data/...|    PARQUET|     2764185|          43256607|
|      0|s3a://warehouse/silver/yellow_taxi_d1a45ac5-5268-493c-818b-acba9f2b22cf/data/...|    PARQUET|     2917644|          45783710|
|      0|s3a://warehouse/silver/yellow_taxi_d1a45ac5-52

## 5. Calidad de datos — visión general

In [36]:
df = spark.table("nessie.silver.yellow_taxi").cache()
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Total records: 57,423,106
Columns: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [37]:
# Registros por año-mes (valida el particionado y la cobertura temporal)
df.groupBy(
    spark_year("tpep_pickup_datetime").alias("year"),
    spark_month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2019|    1|7460359|
|2019|    2|6833264|
|2019|    3|7633194|
|2020|    1|6201962|
|2020|    2|6103376|
|2020|    3|2893173|
|2022|    1|2326489|
|2022|    2|2798864|
|2022|    3|3414499|
|2023|    1|2917644|
|2023|    2|2764185|
|2023|    3|3226937|
|2025|    1|2849160|
+----+-----+-------+



In [38]:
# Nulos por columna. Los 3 últimos campos (congestion_surcharge, airport_fee,
# cbd_congestion_fee) son evolución de esquema NYC TLC: NULL para años previos
# a su introducción por la fuente, no fallo de la limpieza.
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 0        
 trip_distance         | 0        
 RatecodeID            | 0        
 store_and_fwd_flag    | 0        
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 4707240  
 airport_fee           | 37125328 
 cbd_congestion_fee    | 54573946 



In [39]:
# Estadísticas descriptivas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "tolls_amount", "total_amount", "passenger_count"
).summary("count", "min", "max", "mean", "stddev").show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|       tip_amount|      tolls_amount|      total_amount|   passenger_count|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+
|  count|          57423106|          57423106|         57423106|          57423106|          57423106|          57423106|
|    min|               0.0|               0.0|              0.0|               0.0|               0.0|               1.0|
|    max|          62359.52|          943274.8|        141492.02|            3288.0|        1084772.17|               6.0|
|   mean|   3.0244733053973|14.032787563076054|2.466619233383136|0.3903832657519955|20.370920402318205|1.5076177523382313|
| stddev|17.909435678299364|233.28387288652436|18.91794033873774|1.8303189346291782| 277.3682734043688|1.0996552888594529|
+-------+-------

## 6. Validación de las reglas de limpieza

Cada check debe devolver **0 registros**. Si alguno devuelve filas, la regla de limpieza falló.

In [40]:
# Reglas de limpieza consolidadas en una sola pasada sobre df.
# Cada celda devolvía antes una cuenta independiente (1 escaneo por filtro);
# aquí se evalúan todas a la vez y df está cacheado.
VALID_YEARS         = [2019, 2020, 2022, 2023, 2025]
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

checks = {
    "null_pickup":            col("tpep_pickup_datetime").isNull(),
    "null_dropoff":           col("tpep_dropoff_datetime").isNull(),
    "pickup_ge_dropoff":      col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime"),
    "wrong_year":             ~spark_year("tpep_pickup_datetime").isin(VALID_YEARS),
    "neg_trip_distance":      col("trip_distance") < 0,
    "neg_fare_amount":        col("fare_amount")   < 0,
    "neg_total_amount":       col("total_amount")  < 0,
    "neg_tip_amount":         col("tip_amount")    < 0,
    "neg_tolls_amount":       col("tolls_amount")  < 0,
    "invalid_pu_location":    ~col("PULocationID").between(1, 265),
    "invalid_do_location":    ~col("DOLocationID").between(1, 265),
    "invalid_vendor":         ~col("VendorID").isin(VALID_VENDOR_IDS),
    "invalid_ratecode":       ~col("RatecodeID").isin(VALID_RATE_CODES),
    "invalid_payment_type":   ~col("payment_type").isin(VALID_PAYMENT_TYPES),
    "invalid_store_fwd_flag": ~col("store_and_fwd_flag").isin("Y", "N"),
    "invalid_passenger_cnt":  ~col("passenger_count").between(1, 6),
}

results = df.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(label)
    for label, cond in checks.items()
]).collect()[0].asDict()

width = max(len(k) for k in results)
for label, n in results.items():
    flag = "OK" if n == 0 else "FAIL"
    print(f"  [{flag}] {label.ljust(width)} : {n}")

  [OK] null_pickup            : 0
  [OK] null_dropoff           : 0
  [OK] pickup_ge_dropoff      : 0
  [OK] wrong_year             : 0
  [OK] neg_trip_distance      : 0
  [OK] neg_fare_amount        : 0
  [OK] neg_total_amount       : 0
  [OK] neg_tip_amount         : 0
  [OK] neg_tolls_amount       : 0
  [OK] invalid_pu_location    : 0
  [OK] invalid_do_location    : 0
  [OK] invalid_vendor         : 0
  [OK] invalid_ratecode       : 0
  [OK] invalid_payment_type   : 0
  [OK] invalid_store_fwd_flag : 0
  [OK] invalid_passenger_cnt  : 0


In [41]:
# Distribución de los campos categóricos
for field in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag", "passenger_count"]:
    print(f"--- {field} ---")
    df.groupBy(field).count().orderBy(field).show()

--- VendorID ---


+--------+--------+
|VendorID|   count|
+--------+--------+
|       1|18381557|
|       2|39041549|
+--------+--------+

--- RatecodeID ---


+----------+--------+
|RatecodeID|   count|
+----------+--------+
|       1.0|55236091|
|       2.0| 1529295|
|       3.0|  120714|
|       4.0|   57226|
|       5.0|  359635|
|       6.0|     270|
|      99.0|  119875|
+----------+--------+

--- payment_type ---


+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|43909598|
|           2|13136865|
|           3|  224501|
|           4|  152136|
|           5|       6|
+------------+--------+

--- store_and_fwd_flag ---


+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 N|56959221|
|                 Y|  463885|
+------------------+--------+

--- passenger_count ---


+---------------+--------+
|passenger_count|   count|
+---------------+--------+
|            1.0|42582255|
|            2.0| 8592285|
|            3.0| 2250059|
|            4.0| 1051657|
|            5.0| 1832636|
|            6.0| 1114214|
+---------------+--------+



## 7. Comparativa Bronze vs Silver por mes

Muestra cuántas filas se eliminaron en la limpieza, mes a mes.

In [42]:
# Solo meses dentro del scope procesado por silver. Bronze trae filas con pickup en
# años/meses fuera de scope (2001, 2018, 2024, 2088, ...) que ensucian la tabla;
# las filtramos para que la comparativa muestre sólo lo que silver intenta procesar.
VALID_YEARS  = [2019, 2020, 2022, 2023, 2025]
VALID_MONTHS = [1, 2, 3]

def monthly_counts(table: str, branch: str):
    nessie.set_spark_branch(spark, branch)
    return (
        spark.table(table)
        .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
        .filter(spark_month("tpep_pickup_datetime").isin(VALID_MONTHS))
        .groupBy(
            spark_year("tpep_pickup_datetime").alias("year"),
            spark_month("tpep_pickup_datetime").alias("month")
        )
        .agg(count("*").alias(f"{branch}_count"))
    )

bronze_counts = monthly_counts("nessie.bronze.yellow_taxi", "bronze")
silver_counts = monthly_counts("nessie.silver.yellow_taxi", "silver")

comparison = bronze_counts.join(silver_counts, on=["year", "month"], how="outer") \
    .withColumn("dropped", col("bronze_count") - col("silver_count")) \
    .withColumn("retention_pct",
                spark_round(col("silver_count") / col("bronze_count") * 100, 2)) \
    .orderBy("year", "month") \
    .cache()

comparison.show(50)

+----+-----+------------+------------+-------+-------------+
|year|month|bronze_count|silver_count|dropped|retention_pct|
+----+-----+------------+------------+-------+-------------+
|2019|    1|     7696390|     7460359| 236031|        96.93|
|2019|    2|     7049178|     6833264| 215914|        96.94|
|2019|    3|     7866239|     7633194| 233045|        97.04|
|2020|    1|     6405096|     6201962| 203134|        96.83|
|2020|    2|     6299397|     6103376| 196021|        96.89|
|2020|    3|     3007294|     2893173| 114121|        96.21|
|2022|    1|     2463900|     2326489| 137411|        94.42|
|2022|    2|     2979419|     2798864| 180555|        93.94|
|2022|    3|     3627815|     3414499| 213316|        94.12|
|2023|    1|     3066726|     2917644| 149082|        95.14|
|2023|    2|     2914003|     2764185| 149818|        94.86|
|2023|    3|     3403619|     3226937| 176682|        94.81|
|2025|    1|     3475234|     2849160| 626074|        81.98|
|2025|    2|     3577542

In [43]:
# Totales agregados (reutiliza el comparison cacheado en la celda anterior).
nessie.set_spark_branch(spark, "silver")
comparison.select(
    spark_sum("bronze_count").alias("total_bronze"),
    spark_sum("silver_count").alias("total_silver"),
    spark_sum("dropped").alias("total_dropped"),
    spark_round(
        spark_sum("silver_count") / spark_sum("bronze_count") * 100, 2
    ).alias("overall_retention_pct")
).show()

comparison.unpersist()
df.unpersist()

+------------+------------+-------------+---------------------+
|total_bronze|total_silver|total_dropped|overall_retention_pct|
+------------+------------+-------------+---------------------+
|    67977077|    57423106|      2831204|                84.47|
+------------+------------+-------------+---------------------+



DataFrame[VendorID: bigint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: bigint, DOLocationID: bigint, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, cbd_congestion_fee: double]

## 8. Muestra de datos

In [44]:
spark.table("nessie.silver.yellow_taxi").show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2023-01-31 23:57:29 |2023-02-01 00:00:06  |1.0            |0.71         |1.0       |N                 |158         |68  

## 9. Taxi zone lookup — esquema y calidad


In [45]:
zones = spark.table("nessie.silver.taxi_zone_lookup")
print(f"Records: {zones.count()}")
zones.printSchema()
spark.sql("DESCRIBE EXTENDED nessie.silver.taxi_zone_lookup").show(50, truncate=False)


Records: 265
root
 |-- location_id: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                 |comment|
+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [46]:
# Nulos en campos críticos y rango del location_id
print("Null location_id:", zones.filter(col("location_id").isNull()).count())
print("Null borough   :", zones.filter(col("borough").isNull() | (col("borough") == "")).count())
print("Null zone      :", zones.filter(col("zone").isNull() | (col("zone") == "")).count())
print("Out-of-range location_id:",
      zones.filter(~col("location_id").between(1, 265)).count())

# Duplicados por location_id (clave lógica)
dupes = (zones.groupBy("location_id").count()
         .filter(col("count") > 1))
n_dupes = dupes.count()
print("Duplicate location_id:", n_dupes)
if n_dupes > 0:
    dupes.show()

Null location_id: 0
Null borough   : 0
Null zone      : 0
Out-of-range location_id: 0
Duplicate location_id: 0


In [47]:
# Distribución por borough y service_zone
print("--- borough ---")
zones.groupBy("borough").count().orderBy("borough").show(truncate=False)
print("--- service_zone ---")
zones.groupBy("service_zone").count().orderBy("service_zone").show(truncate=False)


--- borough ---
+-------------+-----+
|borough      |count|
+-------------+-----+
|Bronx        |43   |
|Brooklyn     |61   |
|EWR          |1    |
|Manhattan    |69   |
|N/A          |1    |
|Queens       |69   |
|Staten Island|20   |
|Unknown      |1    |
+-------------+-----+

--- service_zone ---
+------------+-----+
|service_zone|count|
+------------+-----+
|Airports    |2    |
|Boro Zone   |205  |
|EWR         |1    |
|N/A         |2    |
|Yellow Zone |55   |
+------------+-----+



In [48]:
# Comparativa Bronze vs Silver (zone lookup es pequeno, ~265 filas: comparamos conjuntos de IDs)
nessie.set_spark_branch(spark, "bronze")
bronze_ids = {row["LocationID"] for row in
              spark.table("nessie.bronze.taxi_zone_lookup").select("LocationID").collect()}
nessie.set_spark_branch(spark, "silver")
silver_ids = {row["location_id"] for row in zones.select("location_id").collect()}

b, s = len(bronze_ids), len(silver_ids)
if b:
    print(f"Bronze: {b}  Silver: {s}  Dropped: {b - s}  Retention: {s / b:.1%}")
else:
    print("Bronze empty")

missing = sorted(bronze_ids - silver_ids)
print(f"IDs in bronze but not in silver: {len(missing)}")
if missing:
    print("  " + ", ".join(str(i) for i in missing))


Bronze: 265  Silver: 265  Dropped: 0  Retention: 100.0%
IDs in bronze but not in silver: 0


In [49]:
# Integridad referencial: ¿todas las PULocationID/DOLocationID de silver.yellow_taxi
# existen en silver.taxi_zone_lookup?
trips = spark.table("nessie.silver.yellow_taxi")

pu_missing = (trips.select(col("PULocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))
do_missing = (trips.select(col("DOLocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))

print("PULocationID not found in zone lookup:", pu_missing.count())
pu_missing.show(truncate=False)
print("DOLocationID not found in zone lookup:", do_missing.count())
do_missing.show(truncate=False)


PULocationID not found in zone lookup: 0


+-----------+
|location_id|
+-----------+
+-----------+



DOLocationID not found in zone lookup: 0


+-----------+
|location_id|
+-----------+
+-----------+



In [50]:
# Muestra
zones.orderBy("location_id").show(10, truncate=False)


+-----------+-------------+-----------------------+------------+
|location_id|borough      |zone                   |service_zone|
+-----------+-------------+-----------------------+------------+
|1          |EWR          |Newark Airport         |EWR         |
|2          |Queens       |Jamaica Bay            |Boro Zone   |
|3          |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4          |Manhattan    |Alphabet City          |Yellow Zone |
|5          |Staten Island|Arden Heights          |Boro Zone   |
|6          |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7          |Queens       |Astoria                |Boro Zone   |
|8          |Queens       |Astoria Park           |Boro Zone   |
|9          |Queens       |Auburndale             |Boro Zone   |
|10         |Queens       |Baisley Park           |Boro Zone   |
+-----------+-------------+-----------------------+------------+
only showing top 10 rows

